# 00 · Configuración del entorno en Google Colab

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

Este notebook prepara todo el entorno para la secuencia de experimentos. Solo
se ejecuta una vez por sesión: monta Google Drive, instala dependencias,
clona el repositorio (si ya existe en GitHub) y crea la estructura de
carpetas persistente.

**Orden de ejecución de la serie:**

| # | Notebook | |
|---|---|---|
| 00 | setup_colab | ← estás aquí |
| 01 | download_reside | descarga de datos |
| 02 | labeling_koschmieder | etiquetas en metros |
| 03 | eda_splits | EDA + split anti-fuga |
| 04 | baseline_resnet50 | baseline |
| 05 | vit_finetune | modelo protagonista |
| 06 | eval_compare | comparación final |

> ⚙️ Antes de continuar: `Entorno de ejecución → Cambiar tipo de entorno de
> ejecución → T4 GPU`.

In [ ]:
# Verificamos GPU disponible
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "Sin GPU: activa T4 en Entorno de ejecución")

In [ ]:
# Dependencias que no vienen preinstaladas en Colab
# (torch y torchvision ya están; timm para el ViT, kaggle para los datos)
!pip install -q timm kaggle
print("Dependencias instaladas")

In [ ]:
# Montamos Google Drive (persistencia de datos, checkpoints y resultados)
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("No estamos en Colab: se usarán rutas locales ./local_workspace")

if IN_COLAB:
    ROOT = Path("/content/drive/MyDrive/camanchaca")
else:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
for p in (DATA_DIR, CKPT_DIR, RESULTS_DIR, FIG_DIR, DATA_DIR / "raw"):
    p.mkdir(parents=True, exist_ok=True)
print(f"Raíz persistente: {ROOT}")

In [ ]:
# Clonamos el repositorio del grupo (dejar el URL real tras publicarlo)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/USUARIO/camanchaca-predict.git"   # <-- completar
REPO_DIR = Path("/content/camanchaca-predict")

if REPO_URL != "https://github.com/USUARIO/camanchaca-predict.git":
    if not REPO_DIR.exists():
        r = subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)])
        if r.returncode == 0:
            print("Repositorio clonado")
else:
    print("Repositorio aún no publicado: los notebooks son autónomos y funcionan igual.")

if REPO_DIR.exists():
    sys.path.insert(0, str(REPO_DIR / "src"))   # permite: from camanchaca... import ...
    print("src/ agregado al path")

In [ ]:
# Semillas y dispositivo (estándar en TODOS los notebooks del proyecto)
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Versiones del entorno (registrar en el reporte para reproducibilidad)
import numpy, pandas, sklearn, matplotlib, PIL, timm
print("numpy     ", numpy.__version__)
print("pandas    ", pandas.__version__)
print("sklearn   ", sklearn.__version__)
print("torch     ", torch.__version__)
print("torchvision", __import__("torchvision").__version__)
print("timm      ", timm.__version__)

## ✅ Checklist antes de continuar

1. **GPU T4 activa** (celda 1 muestra la tarjeta).
2. **Drive montado** y carpeta `camanchaca/` creada.
3. **`kaggle.json` en la raíz de Drive** (kaggle.com → Account → *Create New
   API Token* → guardar el archivo en `Mi unidad/kaggle.json`). Lo pide el
   notebook 01.
4. Repositorio clonado (opcional si aún no está publicado).

Cuando todo esté OK, abre el **notebook 01 · download_reside**.